In [1]:
pip install -U langchain langchain-community langchain-core chromadb sentence-transformers


In [2]:
import wikipediaapi, os, json

# ✅ FIX: specify user_agent explicitly
wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='NurAI-Islamic-Heritage-Explorer/1.0 (contact@example.com)'
)

topics = [
    "Islamic architecture",
    "Ibn Khaldun",
    "House of Wisdom",
    "Alhambra",
    "Islamic art",
    "Fatimid Caliphate",
    "Moorish civilization"
]

os.makedirs("data", exist_ok=True)

for topic in topics:
    page = wiki.page(topic)
    if page.exists():
        with open(f"data/{topic.replace(' ', '_')}.json", "w", encoding="utf-8") as f:
            json.dump({
                "title": topic,
                "url": page.fullurl,
                "text": page.text
            }, f, ensure_ascii=False, indent=2)
        print(f"✅ Saved: {topic}")
    else:
        print(f"⚠️ Skipped: {topic} (page not found)")


✅ Saved: Islamic architecture
✅ Saved: Ibn Khaldun
✅ Saved: House of Wisdom
✅ Saved: Alhambra
✅ Saved: Islamic art
✅ Saved: Fatimid Caliphate
⚠️ Skipped: Moorish civilization (page not found)


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter # Updated import
from langchain_community.vectorstores import Chroma
import json, glob

# Load data
texts, metadatas = [], []
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

for path in glob.glob("data/*.json"):
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
        chunks = splitter.split_text(d["text"])
        for ch in chunks:
            texts.append(ch)
            metadatas.append({"title": d["title"], "url": d["url"]})

# Free embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Store vectors
db = Chroma.from_texts(texts, embeddings, metadatas=metadatas, persist_directory="db")
db.persist()

/tmp/ipython-input-2024825638.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.wa

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipython-input-2024825638.py:23: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [7]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

model_name = "tiiuae/falcon-7b-instruct"

pipe = pipeline(
    "text-generation",
    model=model_name,
    device_map="auto",
    max_new_tokens=512,
    temperature=0.2
)

llm = HuggingFacePipeline(pipeline=pipe)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

Device set to use cpu
/tmp/ipython-input-3580306233.py:14: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [14]:
# Install LangChain packages
!pip install -q langchain langchain-community langchain-core

In [20]:
query = "What was the importance of the House of Wisdom in Islamic civilization?"

retriever = db.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke(query)

print("Documents trouvés:\n")
for i, doc in enumerate(docs, 1):
    print(f"{i}. Source: {doc.metadata.get('title', 'Unknown')}")
    print(f"   Extrait: {doc.page_content[:250]}...\n")

Documents trouvés:

1. Source: House of Wisdom
   Extrait: Other "Houses of Wisdom"
A major contribution from the House of Wisdom in Baghdad is the influence it had on other libraries in the Islamic world. It has been recognised as a factor that connected many different people and empires because of its educ...

2. Source: House of Wisdom
   Extrait: It was destroyed in 1258 during the Mongol siege of Baghdad. The primary sources behind the House of Wisdom narrative date between the late eight centuries and thirteenth centuries, and most importantly include the references in Ibn al-Nadim's (d. 99...

3. Source: House of Wisdom
   Extrait: The House of Wisdom was made possible by the consistent flow of Arab, Persian, and other scholars of the Islamic world to Baghdad, owing to the city's position as capital of the Abbasid Caliphate. This is evidenced by the large number of scholars kno...

